# Supervised Training

This script trains the following supervised models on the ImageNet-100 dataset:
- ResNet-50
- ViT-S/16

## 1. Imports

### 1.1 Install packages

In [ ]:
!pip install timm --quiet

### 1.2 Imports libraries

In [ ]:
import os
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.amp import GradScaler
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
import timm
import shutil
import matplotlib.pyplot as plt

print(timm.__version__)
print(torch.cuda.is_bf16_supported())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print("ok")

## 2. Globals

In [ ]:
SEED = 42

# dataset
IMAGENET_ROOT_TRAIN = "/kaggle/input/datasets/gpucolabkagg/imagenet-100/imagenet100/imagenet1k/train"
IMAGENET_ROOT_VAL = "/kaggle/input/datasets/gpucolabkagg/imagenet-100/imagenet100_val/imagenet1k/val_filtered"

NUM_CLASSES = 100
NUM_WORKERS = min(4, os.cpu_count() or 2)
PREFETCH_FACTOR = 2 # preload batches in advance

# architecture/model
ARCH = "vit_small_patch16_224" # resnet50 or vit_small_patch16_224

# training
EPOCHS = 40 # real objective is 100
# batch size and gradient accumulation
# restnet50: batch 256
# vit-s/16: batch 128 + accum 2 = 256 (vit is more expensive)
BATCH_SIZE = 256 if ARCH == "resnet50" else 128
ACCUM_STEPS = 1 if ARCH == "resnet50" else 2
EFFECTIVE_BS = BATCH_SIZE * ACCUM_STEPS # 256

# optmization
# from https://github.com/galilai-group/lejepa README
LR_BASE = 5e-4
WEIGHT_DECAY = 5e-2 if "vit" in ARCH else 5e-4
WARMUP_EPOCHS = max(4, EPOCHS // 10) # 10% 
ETA_MIN = LR_BASE / 1000
BETAS = (0.9, 0.999)

# mixed precision
USE_AMP = True
AMP_DTYPE = "float16"

# checkpointing
CKPT_DIR = f"/kaggle/working/checkpoints/supervised_{ARCH}"
CKPT_EVERY = 5 # every 5 epochs

# layer hooks for PCA and SAS
HOOK_LAYERS = {
    "resnet50": ["layer1", "layer2", "layer3", "layer4"],
    "vit_small_patch16_224": ["blocks.2", "blocks.5", "blocks.8", "blocks.11"]
}

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()

# cudnn 
cudnn.deterministic = False
cudnn.benchmark = True
  
# checkpoint folder
os.makedirs(CKPT_DIR, exist_ok=True)
if os.path.exists("/kaggle/input/datasets/gpucolabkagg/vit-checkpoints/checkpoints-vit"):
    shutil.copytree(
        "/kaggle/input/datasets/gpucolabkagg/vit-checkpoints/checkpoints-vit",
        CKPT_DIR,
        dirs_exist_ok=True  
    )
print(f"Architecture: {ARCH}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {EFFECTIVE_BS}")
print(f"LR: {LR_BASE} -> {ETA_MIN:.2e} (cosine)")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Warmup epochs: {WARMUP_EPOCHS}")
print(f"AMP dtype: {AMP_DTYPE}")
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Layers for hooks: {HOOK_LAYERS[ARCH]}")
print(f"Device: {DEVICE}")
print(f"GPUs available: {N_GPUS}")

if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", props.total_memory / 1024**3)
    print("Compute capability:", props.major, ".", props.minor, sep="")

print(os.listdir(CKPT_DIR))

## 3. Utils

### 3.1 Seed for reproducibility

In [ ]:
# reproducibility
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(SEED)
print(f"Seed: {SEED}")

### 3.2 Checkpoints

In [ ]:
def save_checkpoint(epoch, model, optimizer, scheduler, scaler, metrics, is_best=False):
    """
    Save checkpoint and best checkpoint
    1. checkpoint_epXXX.pt - checkpoint with all the state dicts
    2. backbone_epXXX.pt - backbone without head
    3. history.json - training history
    """
    _state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    state = {
        "epoch": epoch,
        "arch": ARCH,
        "seed": SEED,
        "model_state": _state,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else {},
        "metrics": metrics,
        "hook_layers": HOOK_LAYERS[ARCH],
        "embed_dim": EMBED_DIM
    }
    path = os.path.join(CKPT_DIR, f"checkpoint_ep{epoch:03d}.pt")
    torch.save(state, path)

    # backbone without head
    backbone_state = {k: v for k, v in _state.items() if not k.startswith("1.")} # 0. = backbone, 1. = head
    backbone_path = os.path.join(CKPT_DIR, f"backbone_ep{epoch:03d}.pt")
    torch.save({
        "backbone_state": backbone_state, "arch": ARCH,
        "embed_dim": EMBED_DIM, "epoch": epoch,
        "seed": SEED, "hook_layers": HOOK_LAYERS[ARCH],
        "val_acc1": metrics.get("val_acc1")
    }, backbone_path)

    if is_best:
      shutil.copy(path, os.path.join(CKPT_DIR, "best_checkpoint.pt"))
      shutil.copy(backbone_path, os.path.join(CKPT_DIR, "best_backbone.pt"))

    print(f"Checkpoint ep{epoch:03d} saved to {path} (best={is_best})")


def load_checkpoint(model, optimizer, scheduler, scaler, ckpt_path):
    """
    Load checkpoint and resume epoch
    """
    state = torch.load(ckpt_path, map_location=DEVICE)
    _model = model.module if hasattr(model, "module") else model 
    _model.load_state_dict(state["model_state"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    if scaler is not None and state.get("scaler"):
        scaler.load_state_dict(state["scaler"])
    resumed_best = state.get("best_acc1") or state["metrics"].get("val_acc1", 0.0)
    print(f"Resumed from epoch {state['epoch']} - val acc1: {state['metrics'].get('val_acc1', 'N/A')} - best so far: {resumed_best:.2f}%")
    return state["epoch"], resumed_best

def find_latest_checkpoint(ckpt_dir):
    """
    Find the checkpoint_epXXX.pt with XXX recent
    """
    if not os.path.isdir(ckpt_dir):
        return None
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("checkpoint_ep") and f.endswith(".pt")]
    if not files:
        return None
    files.sort(key=lambda f: int(f.replace("checkpoint_ep", "").replace(".pt", "")))
    return os.path.join(ckpt_dir, files[-1])

print("ok")

## 4. Data

### 4.1 Upload the data for network training

In [ ]:
# augmentation
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.3, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.RandomApply([
        T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))], p=0.5),
    T.RandomSolarize(threshold=128, p=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# dataset and dataloader
train_dataset = ImageFolder(IMAGENET_ROOT_TRAIN, transform=train_transform)
val_dataset = ImageFolder(IMAGENET_ROOT_VAL, transform=val_transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True, # avoid incomplete batch with AdamW
    persistent_workers=True if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batch")
print(f"Val: {len(val_dataset)} images, {len(val_loader)} batch")
print(f"Classes: {len(train_dataset.classes)}")
print(f"Mapping example: {dict(list(train_dataset.class_to_idx.items())[:3])}")

## 5. Network

### 5.1 Build the network

In [ ]:
def build_model(arch, num_classes):
    """
    Build backbone + liner head
    ResNet50: num_classes = 0 -> output after global average pooling (2048 vector)
    ViT-S/16: global_pool = "token" -> CLS token
    model[0] = backbone
    model[1] = head
    """
    if arch == "resnet50":
        backbone = timm.create_model("resnet50", pretrained=False, num_classes=0)
        embed_dim = backbone.num_features # 2048
    elif arch == "vit_small_patch16_224":
        backbone = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=0, global_pool="token")
        embed_dim = backbone.embed_dim # 384
    else:
        raise ValueError(f"Unknown architecture: {arch}")

    # linear head: embedding -> num_classes
    head = nn.Linear(embed_dim, num_classes)
    nn.init.trunc_normal_(head.weight, std=0.02)
    nn.init.zeros_(head.bias)

    model = nn.Sequential(backbone, head)
    return model, embed_dim


set_seed(SEED)
model, EMBED_DIM = build_model(ARCH, NUM_CLASSES)
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel on {N_GPUS} GPUs")
model.to(DEVICE)

# parameters
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Architecture: {ARCH}")
print(f"Embedding dim: {EMBED_DIM}")
print(f"Number of parameters: {n_params/1e6:.1f}M")
print(f"Number of trainable parameters: {n_train/1e6:.1f}M")

### 5.2 Optimizer and scheduler

In [ ]:
def build_optimizer_and_scheduler(model):
    """
    AdamW + linear warmup + cosine annealing
    lr 5e-4, lr_min = lr/1000, weight decay model specific
    weight decay to non-bias and non-norm parameters only
    """
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(param)  # bias, LayerNorm, BatchNorm
        else:
            decay.append(param)

    param_groups = [
        {"params": decay, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "weight_decay": 0.0}
    ]

    optimizer = optim.AdamW(param_groups, lr=LR_BASE, betas=BETAS)

    # linear warmup
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1e-3, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS-WARMUP_EPOCHS, eta_min=ETA_MIN
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[WARMUP_EPOCHS]
    )
    return optimizer, scheduler

optimizer, scheduler = build_optimizer_and_scheduler(model)
scaler = GradScaler("cuda", enabled=USE_AMP) if AMP_DTYPE == "float16" else None
print(f"Optimizer: {optimizer}")
print(f"Scheduler: {scheduler}")
print(f"Scaler: {scaler}")

## 6. Train

### 6.1 Resume training from checkpoints 

In [ ]:
_resume_ckpt = find_latest_checkpoint(CKPT_DIR)

if _resume_ckpt is not None:
    START_EPOCH, best_acc1 = load_checkpoint(model, optimizer, scheduler, scaler, _resume_ckpt)
    _history_path = os.path.join(CKPT_DIR, "history.json")
    if os.path.exists(_history_path):
        with open(_history_path) as f:
            history = json.load(f)
        # if history recorded epochs next to START EPOCH
        history = [r for r in history if r["epoch"] <= START_EPOCH]
        print(f"Restore History: {len(history)} epochs already registered")
    else:
        history = []
        print("No history.json found")
else:
    START_EPOCH = 0
    best_acc1 = 0.0
    history = []
    print("No checkpoint found: training from scratch")

print(f"Start epoch: {START_EPOCH} | best_acc1: {best_acc1:.2f}%")

### 6.2 Training functions 

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.0)

    total_loss, total_correct, total_samples = 0.0, 0, 0
    optimizer.zero_grad()

    amp_dtype = torch.bfloat16 if AMP_DTYPE == "bfloat16" else torch.float16

    num_batches = len(loader)
    last_accum_steps = num_batches % ACCUM_STEPS  # if not divisible, last step is smaller
    last_batch_idx_to_accum = num_batches - last_accum_steps
    accum_steps = ACCUM_STEPS

    pbar = tqdm(loader, total=num_batches, desc="Training", leave=False)
    for step, (images, labels) in enumerate(pbar):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        last_batch = (step == num_batches - 1)
        if last_accum_steps > 0 and step >= last_batch_idx_to_accum:
            accum_steps = last_accum_steps  # renormalize for last step
        need_update = last_batch or (step + 1) % accum_steps == 0

        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels) / accum_steps

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if need_update:
            if scaler is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()

        with torch.no_grad():
            total_loss += loss.item() * accum_steps * images.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += images.size(0)

        pbar.set_postfix(loss=f"{total_loss/total_samples:.3f}", acc=f"{100*total_correct/total_samples:.1f}%")

    return {
        "loss": total_loss / total_samples,
        "acc1": 100 * total_correct / total_samples
    }


@torch.no_grad()
def validate(model, loader):
    """top-1 and top-5 accuracy on validation set"""
    model.eval()
    criterion = nn.CrossEntropyLoss()

    total_loss, top1_correct, top5_correct, total = 0.0, 0, 0, 0

    amp_dtype = torch.bfloat16 if AMP_DTYPE == 'bfloat16' else torch.float16

    pbar = tqdm(loader, desc="Validation", leave=False)
    for images, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)

        # top-1
        pred1 = logits.argmax(dim=1)
        top1_correct += (pred1 == labels).sum().item()

        # top-5
        _, pred5 = logits.topk(5, dim=1)
        top5_correct += pred5.eq(labels.unsqueeze(1)).any(dim=1).sum().item()

        total += images.size(0)
        pbar.set_postfix(acc1=f"{100*top1_correct/total:.1f}%")

    return {
        "loss": total_loss / total,
        "acc1": 100 * top1_correct / total,
        "acc5": 100 * top5_correct / total
    }

print("ok")

### 6.3 Training loop

In [ ]:
print(f"Start training from epoch {START_EPOCH}/{EPOCHS} - {ARCH}")
print(f"Effective batch size: {EFFECTIVE_BS}")

for epoch in range(START_EPOCH, EPOCHS):
    t0 = time.time()

    # train
    train_metrics = train_one_epoch(model, train_loader, optimizer, scaler)

    # scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # validation
    val_metrics = validate(model, val_loader)

    elapsed = time.time() - t0
    is_best = val_metrics["acc1"] > best_acc1
    if is_best:
        best_acc1 = val_metrics["acc1"]

    # log
    record = {
        "epoch": epoch+1,
        "lr": current_lr,
        "train_loss": train_metrics["loss"],
        "train_acc1": train_metrics["acc1"],
        "val_loss": val_metrics["loss"],
        "val_acc1": val_metrics["acc1"],
        "val_acc5": val_metrics["acc5"],
        "elapsed_s": elapsed
    }
    history.append(record)

    print(
        f"Ep {epoch+1:3d}/{EPOCHS} "
        f"| lr {current_lr:.2e} "
        f"| train loss {train_metrics['loss']:.4f} acc {train_metrics['acc1']:5.1f}% "
        f"| val loss {val_metrics['loss']:.4f} acc1 {val_metrics['acc1']:5.1f}% "
        f"acc5 {val_metrics['acc5']:5.1f}% "
        f"| {'BEST ' if is_best else '      '}"
        f"| {elapsed:.0f}s"
    )

    # checkpoint
    if (epoch + 1) % CKPT_EVERY == 0 or (epoch + 1) == EPOCHS:
        save_checkpoint(epoch+1, model, optimizer, scheduler, scaler, record, is_best=is_best)
        history_path = os.path.join(CKPT_DIR, "history.json")
        with open(history_path, "w") as f:
            json.dump(history, f, indent=4)


# save complete history
history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=4)

print("Training complete!")
print(f"Best accuracy: {best_acc1:.1f}%")
print(f"History saved to {history_path}")


### 6.4 Learning curves

In [ ]:
ep = [r["epoch"] for r in history]
train_loss = [r["train_loss"] for r in history]
val_loss = [r["val_loss"] for r in history]
train_acc = [r["train_acc1"] for r in history]
val_acc = [r["val_acc1"] for r in history]
lrs_hist = [r["lr"] for r in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Training curves - {ARCH} - Supervised - ImageNet100", fontsize=13)

# loss
axes[0].plot(ep, train_loss, label="Train")
axes[0].plot(ep, val_loss, label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# accuracy
axes[1].plot(ep, train_acc, label="Train")
axes[1].plot(ep, val_acc, label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Top-1 Accuracy (%)")
axes[1].set_title("Top-1 Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# LR schedule
axes[2].plot(ep, lrs_hist, color="green")
axes[2].axvline(x=WARMUP_EPOCHS, color="red", linestyle="--", label=f"End warmup (ep {WARMUP_EPOCHS})")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("LR Schedule (warmup + cosine)")
axes[2].set_yscale("log")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CKPT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved in {plot_path}")
print(f"Best val acc1: {max(val_acc):.2f}% (epoch {ep[val_acc.index(max(val_acc))]})")